# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## How to use this worked example

This is an AI-assisted study solution. The original assignment says AI may be used for debugging explanations but not to request solutions; do not present this example as unaided work. All original assignment text is preserved below. Read `README.md` for a step-by-step explanation.

Keep `orders.csv` beside this notebook and run the cells from top to bottom. Tasks 2–8 deliberately use the original rows; cleaning duplicates and excluding TEST happens in tasks 9–10. The saved outputs were computed from the supplied CSV.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Reference answers and explanations (AI-assisted; these are not your own pre-run predictions):

1. `{'A': 2, 'B': 1}`. `get(c, 0)` retrieves the existing count, or zero for a new key, and the loop adds one.
2. `[0, 1, 1]`. Both calls change the same list object using `append`; the function does not create a copy.
3. `2`. Lowercasing produces `almaty`, `almaty`, `astana`, and a set keeps only distinct values.
4. `{'ab': 2, 'abc': 3}`. Dictionary keys are unique; the second `ab` assigns the same value to the existing key.
5. Two lines: `5`, then `None`. The function prints `5` but has no `return`, so its result assigned to `x` is `None`.

The assignment asks for your predictions before running. Use these as study explanations, then explain each fragment yourself.

In [ ]:
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)
# 3

s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))
# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)
# 5

def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)

{'A': 2, 'B': 1}
[0, 1, 1]
2
{'ab': 2, 'abc': 3}
5
None


In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [3]:
import csv
with open("orders.csv", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "rows")

5463 rows


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [ ]:
# 1. Convert text numbers and normalize city names.
def to_num(s):
    text = "".join(s.split()).replace(",", ".")
    if not text:
        return None  #даты которой нет в файле
    return float(text)


def clean_city(s):
    aliases = {
        "алматы": "Алматы", "almaty": "Алматы",
        "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент",
        "караганда": "Караганда", "karaganda": "Караганда",
        "актобе": "Актобе", "aktobe": "Актобе",
        "атырау": "Атырау", "atyrau": "Атырау",
    }
    return aliases[s.strip().lower()]


# 2. каждая строка считается заказом на этом этапе, включая отмененные.
category_counts = {}
for row in rows:
    category = row["category"]
    category_counts[category] = category_counts.get(category, 0) + 1

print("2. Orders by category")
for category, count in sorted(category_counts.items(), key=lambda item: item[1], reverse=True):
    print(f"{category:<15} {count:>6}")

# 3. только оплаченные заказы учитываются в доходе и среднем платном заказе.
category_revenue = {category: 0.0 for category in category_counts}
paid_counts = {category: 0 for category in category_counts}
for row in rows:
    if row["status"] == "оплачен":
        category = row["category"]
        amount = to_num(row["amount"])
        if amount is None:
            raise ValueError("A paid order has no amount: " + row["order_id"])
        category_revenue[category] += amount
        paid_counts[category] += 1

print("\n3. Paid revenue and average order")
print(f"{'Category':<15} {'Revenue':>15} {'Paid orders':>12} {'Average':>12}")
for category in sorted(category_counts):
    count = paid_counts[category]
    average = category_revenue[category] / count if count else None
    average_text = f"{average:,.2f}" if average is not None else "N/A"
    print(f"{category:<15} {category_revenue[category]:>15,.2f} {count:>12} {average_text:>12}")

# 4. YYYY-MM первые 7 символов даты в этом наборе данных.
month_counts = {}
for row in rows:
    month = row["order_date"][:7]
    month_counts[month] = month_counts.get(month, 0) + 1
print("\n4. Orders by month")
for month, count in sorted(month_counts.items()):
    print(month, count)

2. Orders by category
Одежда            1529
Электроника       1185
Дом               1105
Красота            991
Книги              653

3. Paid revenue and average order
Category                Revenue  Paid orders      Average
Дом               10,516,320.00          987    10,654.83
Книги              2,294,100.00          583     3,934.99
Красота            5,507,620.00          890     6,188.34
Одежда            18,321,080.00         1379    13,285.77
Электроника       65,270,240.00         1065    61,286.61

4. Orders by month
2026-09 2782
2026-10 2681


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [5]:
# 5. Reusable functions allow exactly the same calculation before/after cleaning.
def revenue_by_month(data):
    totals = {}
    for row in data:
        month = row["order_date"][:7]
        totals.setdefault(month, 0.0)
        if row["status"] == "оплачен":
            amount = to_num(row["amount"])
            if amount is None:
                raise ValueError("A paid order has no amount: " + row["order_id"])
            totals[month] += amount
    return totals


def percent_change(before, after):
    if before == 0:
        return None  # A percentage relative to zero is undefined.
    return (after - before) / before * 100


monthly_revenue = revenue_by_month(rows)
raw_change = percent_change(monthly_revenue["2026-09"], monthly_revenue["2026-10"])
print("5. Raw paid revenue")
for month, total in sorted(monthly_revenue.items()):
    print(f"{month}: {total:,.2f}")
print(f"October vs September: {raw_change:+.2f}%" if raw_change is not None else "Change: N/A")

# 6. Denominator: all orders in the city; numerator: cancelled orders.
city_counts = {}
cancelled_counts = {}
for row in rows:
    city = clean_city(row["city"])
    city_counts[city] = city_counts.get(city, 0) + 1
    if row["status"] == "отменён":
        cancelled_counts[city] = cancelled_counts.get(city, 0) + 1
print("\n6. Cancellation share by city")
for city in sorted(city_counts):
    share = cancelled_counts.get(city, 0) / city_counts[city] * 100
    print(f"{city:<12} {share:.1f}%")

# 7. Rank paid revenue, not number of orders.
customer_revenue = {}
for row in rows:
    if row["status"] == "оплачен":
        customer = row["customer_id"]
        customer_revenue[customer] = customer_revenue.get(customer, 0) + to_num(row["amount"])
top_customers = sorted(customer_revenue.items(), key=lambda item: (-item[1], item[0]))[:5]
print("\n7. Top five customers")
for customer, total in top_customers:
    print(f"{customer:<10} {total:>15,.2f}")
print("Inspect the leader's orders:")
for row in rows:
    if row["customer_id"] == top_customers[0][0]:
        print(row["order_id"], row["order_date"], row["amount"], row["status"])

# 8. One set per city: repeat purchases do not add another unique customer.
customers_by_city = {}
for row in rows:
    city = clean_city(row["city"])
    customers_by_city.setdefault(city, set()).add(row["customer_id"])
print("\n8. Unique customers by city (all statuses)")
for city, customers in sorted(customers_by_city.items()):
    print(city, len(customers))

5. Raw paid revenue
2026-09: 57,489,850.00
2026-10: 44,419,510.00
October vs September: -22.74%

6. Cancellation share by city
Актобе       10.3%
Алматы       7.9%
Астана       8.8%
Атырау       7.5%
Караганда    8.9%
Шымкент      20.4%

7. Top five customers
TEST         29,999,970.00
C01627          257,180.00
C00485          215,920.00
C01338          192,080.00
C02122          191,670.00
Inspect the leader's orders:
105281 2026-09-12 11:00 9999990.0 оплачен
105282 2026-09-25 15:00 9999990.0 оплачен
105283 2026-10-03 10:00 9999990.0 оплачен

8. Unique customers by city (all statuses)
Актобе 358
Алматы 1351
Астана 981
Атырау 138
Караганда 451
Шымкент 655


### Task 7: what is unusual about the leader?

`TEST` leads with **29,999,970** from only three paid orders, each for **9,999,990**: two in September and one in October. The identifier and repeated extreme amounts suggest test transactions rather than ordinary purchases. The exercise excludes this account; in a real analysis, confirm that interpretation with the data owner before excluding it.

## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [6]:
# 9. Compare every field before normalization; retain the first occurrence.
seen = set()
unique_rows = []
duplicate_counts = {}
for row in rows:
    key = tuple(row.values())
    if key in seen:
        month = row["order_date"][:7]
        duplicate_counts[month] = duplicate_counts.get(month, 0) + 1
    else:
        seen.add(key)
        unique_rows.append(row)
print("9. Extra duplicate rows:", sum(duplicate_counts.values()))
print("Duplicates by month:", duplicate_counts)

# 10. Exclude the explicitly inspected TEST account, not any arbitrary leader.
excluded_customer = "TEST"
clean_rows = [row for row in unique_rows if row["customer_id"] != excluded_customer]
deduplicated_revenue = revenue_by_month(unique_rows)
clean_monthly_revenue = revenue_by_month(clean_rows)
clean_change = percent_change(clean_monthly_revenue["2026-09"], clean_monthly_revenue["2026-10"])
print("\n10. Revenue comparison")
print(f"{'Month':<10} {'Raw':>15} {'Deduplicated':>15} {'No TEST':>15}")
for month in sorted(monthly_revenue):
    print(f"{month:<10} {monthly_revenue[month]:>15,.2f} {deduplicated_revenue[month]:>15,.2f} {clean_monthly_revenue[month]:>15,.2f}")
if raw_change is not None and clean_change is not None:
    print(f"Raw change: {raw_change:+.2f}%; corrected change: {clean_change:+.2f}%")
    print(f"Difference: {clean_change - raw_change:+.2f} percentage points")
else:
    print("Percentage change is undefined when September revenue is zero.")

# 11. The question asks for deduplication only; include every status and TEST.
customer_order_counts = {}
for row in unique_rows:
    customer = row["customer_id"]
    customer_order_counts[customer] = customer_order_counts.get(customer, 0) + 1
maximum = max(customer_order_counts.values(), default=0)
most_orders = sorted((customer, count) for customer, count in customer_order_counts.items() if count == maximum)
print("\n11. Most orders after deduplication:", most_orders)

9. Extra duplicate rows: 180
Duplicates by month: {'2026-09': 180}

10. Revenue comparison
Month                  Raw    Deduplicated         No TEST
2026-09      57,489,850.00   55,243,390.00   35,243,410.00
2026-10      44,419,510.00   44,419,510.00   34,419,520.00
Raw change: -22.74%; corrected change: -2.34%
Difference: +20.40 percentage points

11. Most orders after deduplication: [('C01818', 10)]


### Task 12: note to the manager

After removing exact duplicate rows and the TEST account, October paid revenue fell by 2.34% (from 35,243,410 in September to 34,419,520 in October). The initial calculation suggested a 22.74% decline because September included 180 extra duplicate rows and two TEST orders, while October contained one TEST order. Duplicate paid rows inflated September revenue by 2,246,460, and TEST orders added 19,999,980 in September and 9,999,990 in October. The corrected figures still show a decline, but the raw data substantially overstated its size.